In [86]:
from src import data_loader as dl

df = dl.load_raw_data()
X, y = dl.X_y(df)
time = 'Time'
amt = 'Amount'
print(X.shape, y.shape)

(283726, 30) (283726,)


In [87]:
X_train, X_test, y_train, y_test = dl.split_data(df)
print(X_train.shape, X_test.shape)

(226980, 30) (56746, 30)


### BASELINE MODELS COMPARISON:


In [88]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as imbPipeline
from imblearn.under_sampling import RandomUnderSampler
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from src import config as c

baselines = {
    'LogReg' : LogisticRegression(max_iter=1000 ,random_state=c.RANDOM_STATE),
    'DT' : DecisionTreeClassifier(max_depth=5, random_state=c.RANDOM_STATE),
    'lgbm' : LGBMClassifier(random_state=c.RANDOM_STATE)
}
gaps = {}

def filter(name, model,  type=None):
    scaler = StandardScaler() if name == 'LogReg' else 'passthrough'
    if type is None:
        pipe = Pipeline([
                ('scaler', scaler),
                ('model', model)
            ])
    elif type == 'smote':
        pipe = imbPipeline([
            ('scaler', scaler),
            ('smote', SMOTE(random_state=c.RANDOM_STATE)),
            ('model', model)
        ])
    elif type == 'under':
        pipe = imbPipeline([
            ('scaler', scaler),
            ('undersample', RandomUnderSampler(random_state=c.RANDOM_STATE)),
            ('model', model)
        ])
    return pipe
    
def evaluate(name, model, X_train, X_test, y_train, y_test, type=None):
    pipe = filter(name, model, type)
    pipe.fit(X_train, y_train)
    ytest_pred = pipe.predict(X_test)
    ytrain_pred = pipe.predict(X_train)
    train_score = f1_score(y_train, ytrain_pred)
    test_score = f1_score(y_test, ytest_pred)
    gap = train_score - test_score

    print( '=' * 4, ' OVERFIT CHECK: ', '=' * 4)
    print('TEST F1 score: ', test_score)
    print('TRAIN F1 score: ', train_score)
    print('F1 gap: ', train_score - test_score)
    print(classification_report(y_test, ytest_pred))
    return gap, train_score, test_score

def cvs_pipeline(name, model, X_train, y_train, type=None):
    pipe = filter(name, model, type)
    validation = cross_val_score(pipe, X_train, y_train, cv=c.CV, scoring='f1')
    mean = validation.mean()
    std = validation.std()
    print('=' * 5, ' CROSS-VALIDATION: ', '=' * 5)
    print('mean: ', mean)
    print('std: ', std)
    return mean, std


In [89]:

for name, model in baselines.items():
    print(f'{name} report: ')
    gap, train_score, test_score  = evaluate(name, model, X_train, X_test, y_train, y_test)
    gaps[name] = gap



LogReg report: 
====  OVERFIT CHECK:  ====
TEST F1 score:  0.6875
TRAIN F1 score:  0.71875
F1 gap:  0.03125
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.85      0.58      0.69        95

    accuracy                           1.00     56746
   macro avg       0.92      0.79      0.84     56746
weighted avg       1.00      1.00      1.00     56746

DT report: 
====  OVERFIT CHECK:  ====
TEST F1 score:  0.783625730994152
TRAIN F1 score:  0.8753541076487252
F1 gap:  0.09172837665457323
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.88      0.71      0.78        95

    accuracy                           1.00     56746
   macro avg       0.94      0.85      0.89     56746
weighted avg       1.00      1.00      1.00     56746

lgbm report: 
[LightGBM] [Info] Number of positive: 378, number of negative: 226602
[LightGBM] [Info

In [90]:
for i, j in gaps.items():
    print(f'{i} : {j}')

LogReg : 0.03125
DT : 0.09172837665457323
lgbm : 0.11734084314729476


In [91]:
weighted_baselines = {
    'LogReg' : LogisticRegression(class_weight='balanced' , max_iter=1000 ,random_state=c.RANDOM_STATE),
    'DT' : DecisionTreeClassifier(class_weight='balanced', max_depth=5, random_state=c.RANDOM_STATE),
    'lgbm' : LGBMClassifier(class_weight='balanced', random_state=c.RANDOM_STATE)
}
for name, model in weighted_baselines.items():
    print(f'{name} report: ')
    evaluate(name, model, X_train, X_test, y_train, y_test)


LogReg report: 
====  OVERFIT CHECK:  ====
TEST F1 score:  0.10593490746649649
TRAIN F1 score:  0.11537827591890555
F1 gap:  0.009443368452409062
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56651
           1       0.06      0.87      0.11        95

    accuracy                           0.98     56746
   macro avg       0.53      0.92      0.55     56746
weighted avg       1.00      0.98      0.99     56746

DT report: 
====  OVERFIT CHECK:  ====
TEST F1 score:  0.07431192660550459
TRAIN F1 score:  0.08197088465845465
F1 gap:  0.0076589580529500545
              precision    recall  f1-score   support

           0       1.00      0.96      0.98     56651
           1       0.04      0.85      0.07        95

    accuracy                           0.96     56746
   macro avg       0.52      0.91      0.53     56746
weighted avg       1.00      0.96      0.98     56746

lgbm report: 
[LightGBM] [Info] Number of positive: 378, 

In [92]:
for name, model in weighted_baselines.items():
    print(f'{name} weighted model')
    cvs_pipeline(name, model, X_train, y_train)

LogReg weighted model
=====  CROSS-VALIDATION:  =====
mean:  0.1128794244212276
std:  0.006040644749449831
DT weighted model
=====  CROSS-VALIDATION:  =====
mean:  0.12150215599568423
std:  0.025436117576795918
lgbm weighted model
[LightGBM] [Info] Number of positive: 303, number of negative: 181281
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012587 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 181584, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Info] Number of positive: 303, number of negative: 181281
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012752 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM

## Imbalance Handling — class_weight Comparison

### Baseline (no imbalance handling)

| Model | Test F1 | Train F1 | Gap | CV Mean (±std) |
|---|---|---|---|---|
| LogReg | 0.69 | 0.72 | 0.03 | 0.71 (±0.036) |
| DT | 0.71 | 1.00 | 0.29 | 0.75 (±0.043) |
| LightGBM | 0.47 | 0.55 | 0.08 | 0.33 (±0.056) |

### With `class_weight='balanced'`

| Model | Test F1 | Train F1 | Gap | Precision | Recall |
|---|---|---|---|---|---|
| LogReg | 0.11 | 0.12 | 0.01 | 0.06 | 0.87 |
| DT | 0.07 | 0.08 | 0.01 | 0.04 | 0.85 |
| LightGBM | **0.82** | 0.98 | 0.16 | 0.84 | 0.79 |

**LightGBM CV (weighted):** mean 0.84, std 0.016 — most stable result across all experiments so far.

---

### Key findings

- **LogReg and DT collapsed under class_weight** — recall improved sharply (0.58→0.87, 0.71→0.85) but precision cratered (0.85→0.06, 0.72→0.04). F1 dropped hard because the decision boundary shifted too aggressively toward predicting fraud, without threshold tuning to compensate.
- **LightGBM improved dramatically** — F1 more than doubled (0.47→0.82), precision and recall both landed in a healthy, balanced range (0.84/0.79). Makes sense: boosting's loss function was dominated by the majority class at baseline; class_weight corrects that at the gradient level, not just the decision threshold.
- **LightGBM is now the leading candidate**, reversing the baseline conclusion where LogReg looked like the best generalizer.
- **Some overfitting remains in LightGBM** (train 0.98 vs test 0.82, gap 0.16) — largest gap of the three weighted models. Not disqualifying given the strong test score, but worth addressing via tuning (`num_leaves`, `max_depth`) later.

### Next steps

- [ ] Try SMOTE and undersampling on LightGBM for comparison
- [ ] Revisit LogReg/DT with threshold tuning instead of class_weight alone, before ruling them out
- [ ] If LightGBM remains the winner, move to `tune.py` for hyperparameter search focused on reducing the train/test gap

In [93]:
#smote

model = LGBMClassifier(random_state=c.RANDOM_STATE)

evaluate('LGBMClassifier', model, X_train, X_test, y_train, y_test, type='smote')



[LightGBM] [Info] Number of positive: 226602, number of negative: 226602
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.033834 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 453204, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
====  OVERFIT CHECK:  ====
TEST F1 score:  0.6981132075471698
TRAIN F1 score:  0.9108433734939759
F1 gap:  0.21273016594680616
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.63      0.78      0.70        95

    accuracy                           1.00     56746
   macro avg       0.82      0.89      0.85     56746
weighted avg       1.00      1.00      1.00     56746



(0.21273016594680616, 0.9108433734939759, 0.6981132075471698)

In [94]:
evaluate('LGBMClassifier', model, X_train, X_test, y_train, y_test, type='under')


[LightGBM] [Info] Number of positive: 378, number of negative: 378
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000336 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7564
[LightGBM] [Info] Number of data points in the train set: 756, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wit

(0.014561267175737524, 0.11148798112372807, 0.09692671394799054)

class_weight = 'balance' wins over smote and undersampling methods for imbalanced data handling. Therefore, tunning will be done using LightGBM + class_weight = 'balance'

### MODEL TUNNING